<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents — Multi-Skill Progressive Disclosure
</b></font> </br></p>

---

In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import json
import os
from pathlib import Path

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M34-Multi-Skill"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import check_environment, setup_api_keys, mprint, mermaid, copy_from_github

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"])
check_environment()

In [ ]:
#@title 📦 Installationen { display-mode: "form" }

# Getestete Versionen (Stand: März 2026)
# deepagents 0.4.x: skills=[...]-Parameter ist stabil ab v0.4.10
# ⚠️ Vor einem Update: CHANGELOG prüfen — Sub-Agent-Dict-Syntax kann sich ändern
!uv pip install --system -q "deepagents==0.4.12"

# 1 | Überblick
---

<font color='black' size='5'>Von einem Skill zu vielen</font>

DeepAgents scannt ein Skills-Verzeichnis beim Start, liest die `description`-Felder aus allen `SKILL.md`-Dateien und entscheidet
bei jeder Anfrage dynamisch, welcher Skill passt — **Progressive Disclosure** (schrittweise Offenlegung von Informationen).

| Lernziel | Beschreibung |
|----------|--------------|
| Skills-Scan | Agent liest SKILL.md-Frontmatter automatisch beim Start |
| Skill-Routing | Dynamische Auswahl anhand von Trigger-Phrasen |
| GitHub-Cache | Skill-Dateien aus GitHub laden, lokal cachen, versionieren |
| Abgrenzung | Vergleich mit M33 (manuell) vs. M34 (nativ) |


<p><font color='black' size="5">
Architekturmuster: Skills + Subagents kombinieren
</font></p>

Dieses Modul führt das wichtigste Architekturmuster im DeepAgents-Ökosystem ein:
**Skills liefern das Wissen, Subagents führen die Arbeit isoliert aus.**

Die Rollenverteilung:

| Ebene | Aufgabe | Beispiel in diesem Modul |
|-------|---------|--------------------------|
| **Skill** | Definiert *wie* eine Aufgabe angegangen wird — Ablauf, Regeln, Referenzen | `meeting-briefing/SKILL.md`, `compliance/SKILL.md` |
| **Koordinator-Agent** | Erkennt welcher Skill passt, leitet an Subagent weiter | o3-mini — entscheidet über Routing |
| **Sub-Agent** | Führt die Skill-gesteuerte Aufgabe in *isolierter History* aus | `briefing-writer` (gpt-5.4-mini) |

**Warum Isolation wichtig ist:**

Ohne Subagent-Isolation würde jede Skill-Ausführung die Message-History des Koordinators aufblähen. Bei mehreren Skills und langen Kontext-Dokumenten entsteht schnell Kontext-Overflow. Ein Sub-Agent bekommt nur das, was er für seine Aufgabe braucht.

**Warum Skills und Subagents sich ergänzen:**

Ein Skill ohne Subagent ist möglich — der Koordinator führt dann selbst aus. Ein Subagent ohne Skill ist ebenfalls möglich — er erhält dann nur den `system_prompt`.
Die Kombination entfaltet ihren Mehrwert, wenn:

1. Die Skill-Logik **zu komplex ist für den Koordinator** (viele Schritte, Referenzen, spezifisches Format)
2. Der Koordinator **mehrere Subagenten** für verschiedene Skills hat (dieses Modul)
3. Die Subagenten **unterschiedliche Modelle** benötigen (seit 0.4.11: `model=` im SubAgent-Dict)

```
Anfrage → Koordinator (Skill-Routing) → Sub-Agent A (Skill X ausführen)
                                       → Sub-Agent B (Skill Y ausführen)
```


In [ ]:
#@markdown   <p><font size="4" color='green'>  Skills + Subagents Architekturmuster</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    REQ(["Nutzer-Anfrage"])
    COORD["Koordinator-Agent
o3-mini
Welcher Skill passt?"]
    subgraph SA1 ["Sub-Agent: Compliance-Prüfung"]
        SK1["compliance/SKILL.md
Pflichtschritte · Guardrails · Regeln"]
        W1["Worker
gpt-5.4-mini"]
        SK1 -.->|"informiert"| W1
    end
    subgraph SA2 ["Sub-Agent: Meeting-Briefing"]
        SK2["meeting-briefing/SKILL.md
Abschnitte · Format · Quellenpflicht"]
        W2["Writer
gpt-5.4-mini"]
        SK2 -.->|"informiert"| W2
    end
    REQ --> COORD
    COORD -->|"Compliance-Anfrage"| SA1
    COORD -->|"Briefing-Anfrage"| SA2
    SA1 -->|"Ergebnis"| COORD
    SA2 -->|"Ergebnis"| COORD
    COORD --> ANS(["Antwort"])
    style COORD fill:#E65100,color:#fff
    style SK1   fill:#2E7D32,color:#fff
    style SK2   fill:#2E7D32,color:#fff
    style W1    fill:#1565C0,color:#fff
    style W2    fill:#1565C0,color:#fff
'''
mermaid(diagram, width=750)


In [ ]:
#@markdown   <p><font size="4" color="green">Architektur-Übersicht</font></p>

diagram_arch = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    GH([" 🐙 GitHub Repo\nralf-42/Agenten"]) -->|"copy_from_github()"| CACHE

    subgraph CACHE [" 📁 Lokaler Skills-Cache"]
        direction TB
        S1["compliance/\nSKILL.md · references/ · scripts/"]
        S2["meeting-briefing/\nSKILL.md · references/ · scripts/"]
        S3["research/\nSKILL.md · references/ · scripts/"]
    end

    CACHE -->|"skills=[cache_dir]"| AGENT

    subgraph AGENT [" 🤖 DeepAgent Koordinator — o3"]
        SCAN["Skill-Scan\n(Frontmatter lesen)"] --> INJECT["descriptions →\nSystem-Prompt"]
        INJECT --> ROUTE["Anfrage analysieren\nSkill auswählen"]
    end

    U([" 🧑 Nutzeranfrage"]) --> ROUTE
    ROUTE -->|"compliance"| R1[" ✅ Compliance-Workflow"]
    ROUTE -->|"meeting"| R2[" 📋 Briefing-Workflow"]
    ROUTE -->|"research"| R3[" 🔍 Recherche-Workflow"]
'''
mermaid(diagram_arch, width=700)

# 2 | Setup

---

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends.utils import create_file_data
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from pydantic import BaseModel, Field

from genai_lib.model_config import ROUTER, WORKER  # o3-mini, gpt-5.4-mini

# Koordinator: o3-mini (Demo-Routing, kein temperature)
# o3 wäre zu scharf für diesen Demo-Kontext und hat engeres TPM-Limit (30k)
coord_llm = init_chat_model(ROUTER)

mprint("✅ Pakete geladen")

# 3 | GitHub-Skills laden und lokal cachen

---

Alle `SKILL.md`-Dateien werden aus dem GitHub-Repo `ralf-42/Agenten` geladen
und lokal in `./_skills_cache/` gespeichert.

**Warum lokaler Cache statt direkter GitHub-URL?**

| Aspekt | Direkte URL | Lokaler Cache |
|--------|-------------|---------------|
| Reproduzierbarkeit | ❌ `main` kann sich ändern | ✅ Commit-SHA pinnen |
| Offline-Betrieb | ❌ | ✅ |
| `skills=[...]`-API | ❌ (erwartet lokales FS) | ✅ |
| GitHub-Rate-Limit | ⚠️ bei vielen Runs | ✅ einmal laden |

> 💡 **Commit-SHA pinnen** für vollständige Reproduzierbarkeit:
> `COMMIT_SHA = "abc1234"` statt `"main"`

In [ ]:
# 3.1 Skill-Verzeichnisse komplett aus GitHub laden und lokal cachen

CACHE_DIR = Path("./06_skill")

copy_from_github(
    source="ralf-42/Agenten/06_skill",
    target=str(CACHE_DIR),
)

# Alle Skill-Verzeichnisse auto-entdecken: jedes direkte Unterverzeichnis mit SKILL.md
cached_files = {
    p.parent.name: p
    for p in sorted(CACHE_DIR.rglob("SKILL.md"))
    if p.parent.parent == CACHE_DIR
}
for skill_name, p in cached_files.items():
    mprint(f"✅ `{skill_name}` → `{p.parent}/`")

# 4 | Lokales Skills-Verzeichnis inspizieren

---

Bevor der Agent gebaut wird, zeigen wir explizit, was er beim Start sieht:
die `name`- und `description`-Felder aus dem YAML-Frontmatter jeder `SKILL.md`.

Genau diese `description`-Werte werden in den System-Prompt injiziert,
sodass der Agent bei einer Anfrage erkennt, welcher Skill relevant ist.

In [ ]:
# 4.1 Frontmatter aus SKILL.md-Dateien lesen und anzeigen

import re
import yaml

def read_frontmatter(path: Path) -> dict:
    """Liest YAML-Frontmatter aus einer Markdown-Datei (unterstützt Block-Scalars)."""
    text  = path.read_text(encoding="utf-8")
    match = re.match(r"^---(.+?)---", text, re.DOTALL)
    if not match:
        return {}
    return yaml.safe_load(match.group(1)) or {}

zeilen = ["## 📁 Skills-Verzeichnis: was der Agent beim Start sieht", ""]
for skill_name, p in cached_files.items():
    fm   = read_frontmatter(p)
    name = fm.get("name", "—")
    desc = (fm.get("description", "—") or "—")
    zeilen += [
        f"**`{skill_name}`** &nbsp;·&nbsp; name: `{name}`",
        f"> {desc}",
        "",
    ]
mprint("\n".join(zeilen))

# 5 | DeepAgent mit skills=[...] erzeugen

---

Die `skills=[...]`-API erwartet **lokale Verzeichnispfade**. Der Agent scannt
beim ersten Invoke alle Unterordner nach `SKILL.md`-Dateien und injiziert
deren `description`-Felder in den System-Prompt.

**Wichtig:** `skills_files` übergibt den Datei-Inhalt an das virtuelle Dateisystem
des Agents. Der Pfad-Prefix `/skills/` im Dict muss mit dem `skills=["/skills/"]`-
Parameter übereinstimmen.

```python
# Minimales Muster
agent = create_deep_agent(
    model=llm,
    skills=["/skills/"],   # virtueller FS-Pfad
    checkpointer=InMemorySaver(),
)
result = agent.invoke(
    {"messages": [...], "files": skills_files},
    config={"configurable": {"thread_id": "..."}, "recursion_limit": 50},
)
```

In [ ]:
# 5.1 Einfache Mock-Tools (Demo-Zwecke: zeigt Skill-Routing, nicht volle Logik)

@tool
def compliance_check(subject_name: str, country: str, transaction_amount: float = 0.0) -> str:
    """
    Berechnet den Compliance-Risiko-Score für ein Subjekt.
    Gibt low, medium oder high zurück.
    """
    # Demo: einfache Heuristik — in Produktion: echte Sanktionsdatenbank
    high_risk_countries = {"IR", "KP", "SY", "RU", "BY"}
    risk = "high" if country.upper() in high_risk_countries else (
        "medium" if transaction_amount > 50_000 else "low"
    )
    return json.dumps({"risk_level": risk, "subject": subject_name, "country": country})


@tool
def extract_actions(text: str) -> str:
    """
    Extrahiert Action Items aus Meeting-Text deterministisch.
    Gibt JSON-Liste zurück.
    """
    # Demo: Schlüsselwörter-Heuristik
    actions = []
    keywords = ["bis", "bis zum", "deadline", "verantwortlich", "action"]
    for line in text.split("\n"):
        if any(kw in line.lower() for kw in keywords):
            actions.append({"task": line.strip()[:80], "owner": "[offen]", "due": "[offen]"})
    if not actions:
        actions = [{"task": "Keine Action Items gefunden", "owner": "—", "due": "—"}]
    return json.dumps(actions, ensure_ascii=False)


@tool
def score_relevance(source_text: str, query: str) -> str:
    """
    Bewertet die Relevanz einer Quelle für eine Suchanfrage.
    Gibt Score zwischen 0.0 und 1.0 zurück.
    """
    # Demo: Überlappung von Wörtern als Relevanz-Proxy
    query_words = set(query.lower().split())
    source_words = set(source_text.lower().split())
    overlap = len(query_words & source_words)
    score = min(1.0, overlap / max(len(query_words), 1) * 1.5)
    return json.dumps({"score": round(score, 2), "query": query})


TOOLS = [compliance_check, extract_actions, score_relevance]
mprint(f"✅ {len(TOOLS)} Mock-Tools registriert")

In [ ]:
# 5.2 skills_files-Dict aufbauen (virtuelles Dateisystem — alle Skill-Dateien)

VIRT_PREFIX = "/skills/"  # muss mit skills=[VIRT_PREFIX] übereinstimmen

skills_files = {}
zeilen = []
for skill_name, skill_md_path in cached_files.items():
    skill_dir = skill_md_path.parent
    for fpath in sorted(skill_dir.rglob("*")):
        if fpath.is_file():
            rel       = fpath.relative_to(CACHE_DIR)
            virt_path = f"{VIRT_PREFIX}{rel}"
            skills_files[virt_path] = create_file_data(fpath.read_text(encoding="utf-8"))
            zeilen.append(f"📄 `{virt_path}`")

zeilen.append(f"\n✅ **{len(skills_files)} Dateien** im virtuellen FS registriert")
mprint("\n".join(zeilen))

In [ ]:
# 5.3 DeepAgent mit nativer skills=[...]-API erzeugen

import time

multi_skill_agent = create_deep_agent(
    model=coord_llm,
    tools=TOOLS,
    skills=[VIRT_PREFIX],          # ← native Skills-API
    checkpointer=InMemorySaver(),
    system_prompt=(
        "Du bist ein Allround-Assistent mit Zugriff auf spezialisierte Skills. "
        "Wähle bei jeder Anfrage den passenden Skill aus den verfügbaren Skills. "
        "Wenn kein Skill passt, antworte direkt ohne Skill. "
        "Antworte immer auf Deutsch."
    ),
)


def invoke_agent(anfrage: str, session_id: str = None) -> str:
    """Hilfsfunktion: Agent aufrufen und letzte Antwort zurückgeben."""
    sid = session_id or f"m36-{int(time.time())}"
    result = multi_skill_agent.invoke(
        {
            "messages": [{"role": "user", "content": anfrage}],
            "files":    skills_files,
        },
        config={
            "configurable": {"thread_id": sid},
            "recursion_limit": 50,
            "run_name": f"m36-{sid}",
        },
    )
    return result["messages"][-1].content


mprint("✅ Multi-Skill-Agent bereit")

# 6 | Demo 1 — Compliance-Anfrage

---

**Erwartetes Verhalten:** Der Agent erkennt an der Formulierung
("Lieferantenprüfung", "onboarding", "Risikobewertung"), dass der
`compliance-skill` passt, liest die vollständige `SKILL.md` und
führt den definierten Workflow aus.

> 📌 Progressive Disclosure: Erst die `description` wird gelesen.
> Erst bei einem Match liest der Agent die volle SKILL.md.

In [ ]:
# Demo 1: Compliance — Lieferantenprüfung

anfrage_compliance = """
Ich muss einen neuen Lieferanten onboarden: Firma SteelTech GmbH aus Deutschland,
Auftragswert 35.000 EUR, Bereich Maschinenbauteile.
Sanktionsprüfung wurde durchgeführt, kein Treffer.
Bitte führe eine Risikobewertung durch und gib eine Freigabe-Empfehlung.
"""

mprint(f"**📋 Anfrage:** {anfrage_compliance.strip()}")

antwort_1 = invoke_agent(anfrage_compliance, session_id="demo1-compliance")

mprint(f"**🤖 Agent-Antwort:** {antwort_1}")

# 7 | Demo 2 — Meeting-Briefing-Anfrage

---

**Erwartetes Verhalten:** Trigger-Phrasen wie "Meeting vorbereiten",
"Agenda", "Briefing" matchen den `meeting-briefing-skill`.
Der Skill-Workflow strukturiert die Agenda und extrahiert Action Items.

In [ ]:
# Demo 2: Meeting-Briefing — Sprint-Review vorbereiten

anfrage_meeting = """
Bereite das Sprint-Review für morgen vor.
Teilnehmer: Anna (PO), Ben (Dev), Clara (QA), David (Stakeholder).
Themen: Velocity war 42 SP, 3 Stories offen wegen API-Problemen,
Deployment für Freitag geplant.
Offene Action Items aus dem letzten Meeting: Ben liefert API-Fix bis heute,
Clara erstellt Test-Report bis Donnerstag.
"""

mprint(f"**📋 Anfrage:** {anfrage_meeting.strip()}")

antwort_2 = invoke_agent(anfrage_meeting, session_id="demo2-meeting-briefing")

mprint(f"**🤖 Agent-Antwort:** {antwort_2}")

# 8 | Demo 3 — Mehrdeutige Anfrage mit Skill-Auswahl

---

**Erwartetes Verhalten:** Die Anfrage berührt mehrere Skill-Bereiche.
Der Agent muss entscheiden: Recherche-Skill? Compliance-Skill? Kombination?

Dieses Demo zeigt die **Grenzen und Stärken** des automatischen Routings:
- Bei klaren Trigger-Phrasen → eindeutiger Match
- Bei Überschneidungen → Agent priorisiert oder kombiniert
- Bei keinem Match → direkter Antwort ohne Skill

In [ ]:
# Demo 3: Mehrdeutige Anfrage — Recherche + Compliance-Bezug

anfrage_mixed = """
Ich bereite ein Meeting mit unserem neuen Forschungspartner aus Südkorea vor.
Das Thema ist KI-gestützte Qualitätsprüfung in der Fertigung.
Kannst du mir relevante Hintergrundinformationen zum Thema zusammenstellen
und gleichzeitig checken, ob es besondere Compliance-Anforderungen für
Technologie-Kooperationen mit koreanischen Firmen gibt?
"""

mprint(f"**📋 Anfrage:** {anfrage_mixed.strip()}")

antwort_3 = invoke_agent(anfrage_mixed, session_id="demo3-recherche+compliance")

mprint(f"**🤖 Agent-Antwort:** {antwort_3}")

<p><font color='black' size="5">
🔍 Skill-Routing-Analyse für Demo 3
</font></p>


| Skill | Trigger-Signal in Anfrage | Erwarteter Match |
|-------|--------------------------|------------------|
| compliance | „Compliance-Anforderungen“, „koreanischen Firmen“ | ⚠️ Teilmatch |
| meeting-briefing | „Meeting vorbereiten“, „Hintergrund“ | ⚠️ Teilmatch |
| research | „Hintergrundinformationen“, „zusammenstellen“ | ✅ Starker Match |

> 💡 Bei Überschneidungen wählt der Agent den stärksten Match oder kombiniert Skills — je nach LLM-Entscheidung. Das Verhalten ist nicht deterministisch und kann variieren.

# 9 | Vergleich: natives Skill-System vs. M33-Ansatz

---

Beide Ansätze sind didaktisch wertvoll und ergänzen sich:

| Dimension | M33 (manuell, Gate-Writer) | M34 (nativ, Progressive Disclosure) |
|-----------|--------------------------|--------------------------------------|
| **Skill-Loading** | Explizit via `urllib.request` | Automatisch via `skills=[...]` |
| **Skill-Quelle** | GitHub Raw URL (direkt) | GitHub → lokaler Cache |
| **Routing** | Fest verdrahtet (ein Skill) | Dynamisch durch LLM |
| **Anzahl Skills** | 1 (meeting-briefing) | 3 (compliance, meeting, research) |
| **Gate-Writer** | ✅ Vollständig (o3 → gpt-5.1) | ❌ Nicht implementiert |
| **Kontrolle** | ✅ Sehr hoch | ⚠️ Mittel (LLM entscheidet) |
| **Transparenz** | ✅ Jeder Schritt sichtbar | ⚠️ Routing ist black-box |
| **Erweiterbarkeit** | Aufwendig (neuer Code) | ✅ Neuer Ordner + SKILL.md |
| **Reproduzierbarkeit** | ✅ Hoch (SHA-pinned) | ✅ Hoch (SHA-pinned + Version) |

**Wann welchen Ansatz wählen?**

```
Genau ein Skill, kritische Logik, Gate-Writer-Qualität   → M33-Ansatz
Mehrere Skills, flexible Erweiterung, Rapid Prototyping  → M34-Ansatz
Produktion mit Audit-Trail                               → M33-Ansatz
Interne Tools / Wissensdatenbank                         → M34-Ansatz
```

In [ ]:
#@markdown   <p><font size="4" color="green">Entscheidungsbaum: M33 vs. M34</font></p>

diagram_compare = '''
%%{init: {'theme':'light'}}%%
flowchart TB
    START([" ❓ Neues Skill-System bauen"]) --> Q1{Wie viele Skills?}

    Q1 -->|"genau 1"| Q2{Gate-Writer
Qualität nötig?}
    Q1 -->|"> 1"| Q3{Routing flexibel
bzw. erweiterbar?}

    Q2 -->|"ja"| M33[" 🔵 M33-Ansatz\nManuelle Kontrolle"]
    Q2 -->|"nein"| M34A[" 🟢 M34-Ansatz\nEinfacher"]

    Q3 -->|"ja"| M34B[" 🟢 M34-Ansatz\nProgressive Disclosure"]
    Q3 -->|"nein"| Q4{Audit-Trail
erforderlich?}

    Q4 -->|"ja"| M33
    Q4 -->|"nein"| M34B
'''
mermaid(diagram_compare, width=550)

# 10 | Grenzen, Stabilität, didaktische Einordnung

---

## Bekannte Grenzen

| Grenze | Details |
|--------|---------|
| **Lokales FS erforderlich** | `skills=[...]` akzeptiert keine GitHub-URLs direkt — Cache-Schritt ist immer nötig |
| **Nicht deterministisch** | LLM-Routing kann bei ähnlichen Trigger-Phrasen variieren |
| **Kein Gate-Writer** | Natives Skill-System hat keine eingebaute zweistufige Analyse |
| **Versions-API** | `skills=`-Parameter ist in v0.4 noch nicht als stable markiert |
| **Debugging** | Schwieriger als M33: Welcher Skill gewählt wurde, ist nicht direkt sichtbar |

## Stabilität (Stand: März 2026)

```python
# Getestete, stabile Kombination
deepagents==0.4.12
langchain>=1.0.0
langgraph>=1.0.0
```

> ⚠️ **Vor einem Versions-Update:** `/check-deepagents`-Command ausführen
> und das M34-CHANGELOG überprüfen.

## Didaktische Einordnung im Kurs

```
M31  Agent-Skill-Compliance     → Skill-Konzept einführen
M32  DeepAgents-Harness         → Harness-Grundlagen
M33  Meeting-Briefing-Skill     → Skill-Tiefe: Gate-Writer, explizite Kontrolle
M34  Multi-Skill Progressive    → Skill-Breite: Routing, Erweiterbarkeit
```

M33 und M34 sind **keine Alternativen**, sondern **komplementäre Lernpfade**.
Beide zusammen ergeben das vollständige Bild moderner Skill-basierter Agenten.